# DỰ ÁN 1 - XÂY DỰNG MÔ HÌNH PHÂN LOẠI VÀ DỰ BÁO RỦI RO KHÁCH HÀNG VAY VỐN

**Notebook 02/07 - Database Pipeline trung tâm (PostgreSQL)**

---

**Mục tiêu:** Tạo database PostgreSQL làm nguồn dữ liệu trung tâm cho dự án: tạo bảng raw, import CSV, tạo view/join/aggregation, kiểm tra dữ liệu sau import và sau aggregation.

**Input:** `data/raw/*.csv`; các script `sql/01_create_tables.sql` -> `sql/05_indexes.sql`.

**Output:** Database PostgreSQL gồm bảng raw, view nghiệp vụ, materialized view tổng hợp và hợp đồng bảng/view để NB03, NB04, NB05, NB06 đọc bằng `pd.read_sql`.

**Pipeline:** 01. Data Understanding -> **02. Database Pipeline** -> NB03/NB04/NB05/NB06 đọc dữ liệu từ PostgreSQL khi cần.


## Chuẩn bị trước khi chạy

Sau khi `git pull`, notebook này cần chuẩn bị trên từng máy: cài thư viện, bật PostgreSQL, tạo database `credit_risk_db`, tạo `.env`, và đặt 8 file CSV vào `data/raw/`.

Chạy lại notebook nhiều lần vẫn được vì bước import có `TRUNCATE` trước khi `COPY`.


## 1. Vai trò của NB02 trong pipeline

NB02 là nơi gom phần database của dự án. Các notebook sau không nên tự đoán tên bảng hoặc tự viết lại join lớn nếu dữ liệu đó đã có ở database.

| Loại dữ liệu | Nơi tạo | Notebook đọc tiếp |
|---|---|---|
| Bảng raw (`application_train`, `bureau`, ...) | NB02 | NB03, NB04, NB05 |
| View nghiệp vụ (`v_*`) | NB02 qua `sql/03_views.sql` | NB04, NB05 |
| Bảng tổng hợp (`agg_*`) | NB02 qua `sql/04_aggregation.sql` | NB04, NB05 |
| Bảng clean (`application_*_clean`) | NB03 ghi ngược về DB | NB04, NB05, NB06 |
| Bảng features (`train_features`, `test_features`) | NB05 nên ghi ngược về DB | NB06, NB07, app |

Điểm chính: dữ liệu raw đi vào database trước; dữ liệu clean/features tạo ở Python thì cũng ghi lại database để cả nhóm đọc cùng một nguồn.


## 2. Chuẩn bị môi trường và kết nối database


Đoạn code bên dưới nạp thư viện và đọc cấu hình kết nối database.


In [1]:
import os
import time
import psycopg2
from psycopg2 import sql
import pandas as pd
from dotenv import load_dotenv, find_dotenv

# Load file .env cấu hình database kết nối
# Tự động tìm kiếm file .env ở thư mục gốc bất kể thư mục chạy của Jupyter.
# override=True để luôn nạp lại giá trị mới nhất từ .env, tránh dùng nhầm
# biến môi trường cũ còn sót trong kernel (nguyên nhân lỗi auth khi kernel chưa restart).
load_dotenv(find_dotenv(), override=True)

# Lấy thông số kết nối từ biến môi trường
db_host = os.getenv('DB_HOST', 'localhost')
db_port = os.getenv('DB_PORT', '5432')
db_name = os.getenv('DB_NAME', 'credit_risk_db')
db_user = os.getenv('DB_USER', 'postgres')
db_password = os.getenv('DB_PASSWORD', 'postgres')

print("Thông số kết nối database:")
print(f"- Host: {db_host}")
print(f"- Port: {db_port}")
print(f"- Database: {db_name}")
print(f"- User: {db_user}")

Thông số kết nối database:
- Host: localhost
- Port: 5432
- Database: credit_risk_db
- User: postgres


Đoạn code bên dưới tạo hàm chạy một file SQL.


In [2]:
def execute_sql_file(file_path, conn):
    """Hàm đọc và thực thi toàn bộ câu lệnh trong một file SQL"""
    start_time = time.time()
    with open(file_path, 'r', encoding='utf-8') as f:
        sql_script = f.read()
    
    try:
        with conn.cursor() as cursor:
            cursor.execute(sql_script)
        conn.commit()
        elapsed = time.time() - start_time
        print(f"Đã chạy file: {file_path} ({elapsed:.2f} giây).")
    except Exception as e:
        conn.rollback()
        print(f"Lỗi khi thực thi file {file_path}: {e}")
        raise e

## 3. Tạo bảng raw trong PostgreSQL


Đoạn code bên dưới mở kết nối PostgreSQL và tạo bảng raw.


In [3]:
# Khởi tạo kết nối tới database
conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    database=db_name,
    user=db_user,
    password=db_password
)

# Chạy file tạo bảng thô
execute_sql_file('../sql/01_create_tables.sql', conn)

Đã chạy file: ../sql/01_create_tables.sql (0.48 giây).


## 4. Import dữ liệu raw vào database

Dữ liệu raw vẫn giữ nguyên ở database để đối chiếu lại khi cần. Notebook dùng `COPY FROM STDIN` vì cách này ổn trên Windows.


Đoạn code bên dưới tạo hàm import CSV vào PostgreSQL bằng `COPY`.


In [4]:
def import_csv_to_db(table_name, csv_path, conn):
    """Hàm import file CSV vào bảng tương ứng sử dụng copy_expert.

    TRUNCATE bảng ngay trước khi COPY để hàm chạy lại được nhiều lần mà
    KHÔNG bị nạp trùng dữ liệu (idempotent). TRUNCATE và COPY nằm chung
    một transaction: nếu COPY lỗi thì rollback cả hai -> bảng giữ nguyên
    dữ liệu cũ, không rơi vào trạng thái rỗng dở dang.
    """
    start_time = time.time()
    print(f"Đang import dữ liệu vào bảng '{table_name}' từ file '{csv_path}'...")
    
    # Lệnh COPY FROM STDIN đọc luồng dữ liệu từ Client
    copy_query = f"COPY {table_name} FROM STDIN WITH (FORMAT CSV, HEADER, DELIMITER ',')"
    
    try:
        with conn.cursor() as cursor:
            # Xóa sạch dữ liệu cũ trước khi nạp -> tránh cộng dồn khi chạy lại.
            # Dùng sql.Identifier để đóng ngoặc kép tên bảng an toàn.
            cursor.execute(
                sql.SQL("TRUNCATE TABLE {} RESTART IDENTITY").format(
                    sql.Identifier(table_name)
                )
            )
            with open(csv_path, 'r', encoding='utf-8') as f:
                cursor.copy_expert(copy_query, f)
        conn.commit()
        elapsed_time = time.time() - start_time
        print(f"Đã import bảng '{table_name}' trong {elapsed_time:.2f} giây.\n")
    except Exception as e:
        conn.rollback()
        print(f"Lỗi khi import bảng '{table_name}': {e}\n")
        raise e

Đoạn code bên dưới import 8 file CSV thô vào database.


In [5]:
# Danh sách mapping giữa tên bảng trong DB và đường dẫn file CSV thô
csv_mapping = {
    'application_train': '../data/raw/application_train.csv',
    'application_test': '../data/raw/application_test.csv',
    'bureau': '../data/raw/bureau.csv',
    'bureau_balance': '../data/raw/bureau_balance.csv',
    'previous_application': '../data/raw/previous_application.csv',
    'pos_cash_balance': '../data/raw/POS_CASH_balance.csv',
    'installments_payments': '../data/raw/installments_payments.csv',
    'credit_card_balance': '../data/raw/credit_card_balance.csv'
}

# Thực hiện import tự động
total_start = time.time()
for table, path in csv_mapping.items():
    if os.path.exists(path):
        import_csv_to_db(table, path, conn)
    else:
        print(f"Không tìm thấy file CSV thô tại: {path}\n")

print(f"Hoàn thành import 8 bảng dữ liệu thô trong {time.time() - total_start:.2f} giây!")

Đang import dữ liệu vào bảng 'application_train' từ file '../data/raw/application_train.csv'...
Đã import bảng 'application_train' trong 9.18 giây.

Đang import dữ liệu vào bảng 'application_test' từ file '../data/raw/application_test.csv'...
Đã import bảng 'application_test' trong 2.05 giây.

Đang import dữ liệu vào bảng 'bureau' từ file '../data/raw/bureau.csv'...
Đã import bảng 'bureau' trong 19.79 giây.

Đang import dữ liệu vào bảng 'bureau_balance' từ file '../data/raw/bureau_balance.csv'...
Đã import bảng 'bureau_balance' trong 165.17 giây.

Đang import dữ liệu vào bảng 'previous_application' từ file '../data/raw/previous_application.csv'...
Đã import bảng 'previous_application' trong 34.82 giây.

Đang import dữ liệu vào bảng 'pos_cash_balance' từ file '../data/raw/POS_CASH_balance.csv'...
Đã import bảng 'pos_cash_balance' trong 29.66 giây.

Đang import dữ liệu vào bảng 'installments_payments' từ file '../data/raw/installments_payments.csv'...
Đã import bảng 'installments_payment

## 5. Tạo view, join, aggregation và index

Các phép join/aggregation lớn nên nằm ở PostgreSQL. Python sẽ đọc kết quả đã gọn bằng `pd.read_sql` thay vì kéo toàn bộ bảng lịch sử về RAM.


Đoạn code bên dưới tạo view, materialized view và index.


In [6]:
print("1. Đang tạo các View phân tích...")
execute_sql_file('../sql/03_views.sql', conn)

print("2. Đang tạo các bảng tổng hợp (Materialized Views)...(Việc này có thể mất vài chục giây do phải tính toán trên lượng dữ liệu lớn)")
execute_sql_file('../sql/04_aggregation.sql', conn)

print("3. Đang tạo các chỉ mục tối ưu hóa hiệu năng JOIN (Indexes)...")
execute_sql_file('../sql/05_indexes.sql', conn)

1. Đang tạo các View phân tích...
Đã chạy file: ../sql/03_views.sql (0.15 giây).
2. Đang tạo các bảng tổng hợp (Materialized Views)...(Việc này có thể mất vài chục giây do phải tính toán trên lượng dữ liệu lớn)
Đã chạy file: ../sql/04_aggregation.sql (95.48 giây).
3. Đang tạo các chỉ mục tối ưu hóa hiệu năng JOIN (Indexes)...
Đã chạy file: ../sql/05_indexes.sql (89.39 giây).


## 6. Hợp đồng dữ liệu cho các notebook sau

Bảng dưới đây là danh sách tên bảng/view mà nhóm nên dùng thống nhất khi đọc bằng `pd.read_sql`.


Đoạn code bên dưới liệt kê bảng/view cho các notebook sau đọc bằng `pd.read_sql`.


In [ ]:
DATASETS_FOR_NOTEBOOKS = [
    ["application_train", "raw table", "NB03, NB05", "pd.read_sql('SELECT * FROM application_train', conn)", "Có sau NB02"],
    ["application_test", "raw table", "NB03, NB05, NB07/app", "pd.read_sql('SELECT * FROM application_test', conn)", "Có sau NB02"],
    ["v_application_all", "view", "NB03 nếu muốn clean chung train/test", "pd.read_sql('SELECT * FROM v_application_all', conn)", "Có sau NB02"],
    ["v_bureau_clean", "view", "NB04, NB05", "pd.read_sql('SELECT * FROM v_bureau_clean', conn)", "Có sau NB02"],
    ["v_previous_application_clean", "view", "NB04, NB05", "pd.read_sql('SELECT * FROM v_previous_application_clean', conn)", "Có sau NB02"],
    ["agg_installments_summary", "materialized view", "NB04, NB05", "pd.read_sql('SELECT * FROM agg_installments_summary', conn)", "Có sau NB02"],
    ["agg_pos_cash_summary", "materialized view", "NB04, NB05", "pd.read_sql('SELECT * FROM agg_pos_cash_summary', conn)", "Có sau NB02"],
    ["agg_credit_card_summary", "materialized view", "NB04, NB05", "pd.read_sql('SELECT * FROM agg_credit_card_summary', conn)", "Có sau NB02"],
    ["application_train_clean / application_test_clean", "clean table", "NB04, NB05, NB06", "pd.read_sql('SELECT * FROM application_train_clean', conn)", "NB03 ghi ngược về DB"],
    ["train_features / test_features", "feature table", "NB06, NB07, app", "pd.read_sql('SELECT * FROM train_features', conn)", "NB05 nên ghi ngược về DB"],
]

pd.DataFrame(
    DATASETS_FOR_NOTEBOOKS,
    columns=["Tên bảng/view", "Loại", "Notebook đọc", "Cách đọc gợi ý", "Trạng thái"],
)


## 7. Validation sau import và sau aggregation

Validation chia làm hai nhóm: số dòng raw phải khớp NB01; view/materialized view phải tồn tại; bảng tổng hợp không được nhân dòng.


Đoạn code bên dưới đối chiếu số dòng raw sau import.


In [ ]:
expected_raw_rows = {
    "application_train": 307_511,
    "application_test": 48_744,
    "bureau": 1_716_428,
    "bureau_balance": 27_299_925,
    "previous_application": 1_670_214,
    "pos_cash_balance": 10_001_358,
    "installments_payments": 13_605_401,
    "credit_card_balance": 3_840_312,
}

raw_checks = []
with conn.cursor() as cursor:
    for table_name, expected in expected_raw_rows.items():
        cursor.execute(sql.SQL("SELECT COUNT(*) FROM {}").format(sql.Identifier(table_name)))
        actual = cursor.fetchone()[0]
        raw_checks.append({
            "Bảng raw": table_name,
            "Số dòng mong đợi": expected,
            "Số dòng thực tế": actual,
            "Kết quả": "OK" if actual == expected else "Cần kiểm tra",
        })

pd.DataFrame(raw_checks)


Đoạn code bên dưới kiểm tra view/materialized view và khóa sau aggregation.


In [ ]:
object_names = [
    "v_application_all", "v_bureau_clean", "v_bureau_balance_summary",
    "v_previous_application_clean", "v_installments_detail", "v_pos_cash_clean",
    "v_credit_card_clean", "agg_installments_summary", "agg_pos_cash_summary",
    "agg_credit_card_summary",
]

with conn.cursor() as cursor:
    object_checks = []
    for object_name in object_names:
        cursor.execute("SELECT to_regclass(%s)", (object_name,))
        exists = cursor.fetchone()[0] is not None
        row_count = None
        if exists:
            cursor.execute(sql.SQL("SELECT COUNT(*) FROM {}").format(sql.Identifier(object_name)))
            row_count = cursor.fetchone()[0]
        object_checks.append({"Bảng/View": object_name, "Tồn tại": exists, "Số dòng": row_count})

    cursor.execute("""
        SELECT is_train, COUNT(*) AS row_count
        FROM v_application_all
        GROUP BY is_train
        ORDER BY is_train DESC
    """)
    app_all_split = pd.DataFrame(cursor.fetchall(), columns=["is_train", "Số dòng"])

    unique_checks = []
    for object_name in ["agg_installments_summary", "agg_pos_cash_summary", "agg_credit_card_summary"]:
        cursor.execute(sql.SQL("""
            SELECT COUNT(*) AS row_count, COUNT(DISTINCT sk_id_curr) AS distinct_customer
            FROM {}
        """).format(sql.Identifier(object_name)))
        row_count, distinct_customer = cursor.fetchone()
        unique_checks.append({
            "Bảng tổng hợp": object_name,
            "Số dòng": row_count,
            "Số khách duy nhất": distinct_customer,
            "Kết quả": "OK" if row_count == distinct_customer else "Bị nhân dòng",
        })

print("Phân tách train/test trong v_application_all:")
display(app_all_split)
print("Kiểm tra view/materialized view:")
display(pd.DataFrame(object_checks))
print("Kiểm tra khóa sau aggregation:")
pd.DataFrame(unique_checks)


## 8. Đóng kết nối database


Đoạn code bên dưới đóng kết nối PostgreSQL.


In [8]:
if conn:
    conn.close()
    print("Đã đóng kết nối PostgreSQL.")

Đã đóng kết nối PostgreSQL.


## 9. Tổng kết

Trong notebook này, nhóm đã chuyển NB02 thành **Database Pipeline trung tâm**:

1. Tạo 8 bảng raw trong PostgreSQL.
2. Import toàn bộ CSV raw bằng `COPY`.
3. Tạo view nghiệp vụ, materialized view tổng hợp và index.
4. Ghi rõ hợp đồng dữ liệu để NB03, NB04, NB05, NB06 đọc bằng `pd.read_sql`.
5. Thêm validation sau import và sau aggregation.

**Bước tiếp theo:** NB03 làm sạch có chủ đích, xuất CSV như cũ và ghi bảng clean về PostgreSQL nếu máy đã cấu hình `.env`.
